In [1]:
import torch
import torchvision
import torch.nn as nn
import numpy as np
import torchvision.transforms as transforms # contains preprocessing functions

In [2]:
# Create tensors
# required_grad = True - track operations performed on this tensor; useful for backpropagation
x = torch.tensor(1., requires_grad=True)
w = torch.tensor(2., requires_grad=True)
b = torch.tensor(3., requires_grad=True)

# Build a computational graph
y = w * x + b

# Compute gradients; performs backpropagation
y.backward()

# Print the gradients.
print(x.grad)
print(w.grad)
print(b.grad)

tensor(2.)
tensor(1.)
tensor(1.)


In [3]:
# Create tensors of shape (10, 3) and (10, 2)
x = torch.randn(10, 3)
y = torch.randn(10, 2)

# Build a fully connected layer
linear = nn.Linear(3, 2) # 3 input neurons, 2 output neurons
print ('w: ', linear.weight)
print ('b: ', linear.bias)

# Build loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(linear.parameters(), lr=0.01)

# Forward pass
pred = linear(x)

# Compute loss
loss = criterion(pred, y)
print('loss: ', loss.item())

# Backward pass
loss.backward()

# Print the gradients
print ('dL/dw: ', linear.weight.grad)
print ('dL/db: ', linear.bias.grad)

# 1-step gradient descent; update the weights
optimizer.step()

# Print the loss after 1-step gradient descent
pred = linear(x)
loss = criterion(pred, y)
print('loss after 1 step optimization: ', loss.item())

w:  Parameter containing:
tensor([[-0.3369, -0.5415, -0.4921],
        [-0.5355, -0.0812,  0.3450]], requires_grad=True)
b:  Parameter containing:
tensor([0.2526, 0.1469], requires_grad=True)
loss:  1.4588221311569214
dL/dw:  tensor([[ 0.1631, -0.9709, -0.0701],
        [-0.4480,  0.2331,  0.8213]])
dL/db:  tensor([0.1025, 1.0562])
loss after 1 step optimization:  1.4287528991699219


In [4]:
# Create a numpy array
x = np.array([[1, 2], [3, 4]])

# Convert the numpy array to a torch tensor
y = torch.from_numpy(x)

# Convert the torch tensor to a numpy array
z = y.numpy()

In [6]:
# Download and construct CIFAR-10 dataset
# converts the PIL image to tensor
train_dataset = torchvision.datasets.CIFAR10(root='../../data/',
                                             train=True,
                                             transform=transforms.ToTensor(),
                                             download=True)

# Read first sample
image, label = train_dataset[0]
print(image.size())  # output: [channels, pixel, pixel]
print(label)

# Data loader - batch by batch, randomize order every epoch
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=64,
                                           shuffle=True)

# makes data loader iterable
data_iter = iter(train_loader)

# gets one batch
images, labels = data_iter.next()

# training loop
for images, labels in train_loader:
    # Training code should be written here.
    # load batch -> forward pass -> compute loss -> backward pass -> update weights
    pass


In [ ]:
# Build custom dataset
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self):
        # TODO
        # 1. Initialize file paths or a list of file names
        pass

    def __getitem__(self, index):
        # TODO
        # 1. Read one data from file (e.g. using numpy.fromfile, PIL.Image.open)
        # 2. Preprocess the data (e.g. torchvision.Transform)
        # 3. Return a data pair (e.g. image and label)
        pass

    def __len__(self):
        # You should change 0 to the total size of your dataset
        return 0

# use the prebuilt data loader
custom_dataset = CustomDataset()
train_loader = torch.utils.data.DataLoader(dataset=custom_dataset,
                                           batch_size=64,
                                           shuffle=True)

In [ ]:
# Download and load the pretrained ResNet-18.
resnet = torchvision.models.resnet18(pretrained=True)

# freeze the initial layers
for param in resnet.parameters():
    param.requires_grad = False

# replace the top layer for finetuning
resnet.fc = nn.Linear(resnet.fc.in_features, 100)  # eg.100

# Forward pass
images = torch.randn(64, 3, 224, 224)
outputs = resnet(images)
print (outputs.size())     # (64 predictions, 100 classes)


In [ ]:
# Save and load the entire model
torch.save(resnet, 'model.ckpt') # stores architecture, weights
model = torch.load('model.ckpt') # restore everything

# Save and load only the model parameters (recommended).
torch.save(resnet.state_dict(), 'params.ckpt') # store only learned parameters - weights, biases
resnet.load_state_dict(torch.load('params.ckpt')) # load the weights